In [ ]:
from nltk.tokenize import TweetTokenizer
from sklearn.model_selection import cross_val_score, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import VotingClassifier
from sklearn.svm import SVC
from sklearn import metrics
import numpy as np
import os

# Load data parsing function
import sys
sys.path.insert(0, '../..')
sys.path.insert(0, '..')
from load import parse_dataset
from slovene_pipeline.features import build_w2v_mean
from word2vec_api import Word2VecAPI, maybe_load_emoji2vec

In [ ]:
# Experiment settings

DATASET_FP = "../../datasets/slovene/train/SemEval2018-T3-train-taskA_emoji.txt"
TASK = "A"
FNAME = './predictions-task' + TASK + '.txt'
PREDICTIONSFILE = open(FNAME, "w")

K_FOLDS = 10 # 10-fold crossvalidation

random_state = 11

# IMPROVEMENT #1: Class weight balancing in SVM
CLF1 = SVC(random_state=random_state, probability=True, class_weight='balanced')
CLF2 = LogisticRegression(random_state=random_state, n_jobs=-1) 
CLF = VotingClassifier(estimators=[('svm', CLF1), ('lr', CLF2)], voting='soft', n_jobs=-1)

# Loading dataset 
corpus, y = parse_dataset(DATASET_FP)

print("Loading word2vec and emoji models...")
import gensim

# Load FastText model and extract KeyedVectors
kv = gensim.models.fasttext.load_facebook_model('../../w2v-slo/all-token-prelim.ft.sg.bin').wv
w2v_model = Word2VecAPI(kv)
emoji_model = maybe_load_emoji2vec('../../emoji2vec-master/pre-trained/emoji2vec.bin', binary=True)

print("Building w2v+emoji features for training...")
w2v_features = build_w2v_mean(corpus, w2v_model, emoji_model)
print(f"W2V features shape: {w2v_features.shape}")

print("Loading OG handcrafted features for training...")
# Load handcrafted features generated by feature_generator_TaskA_og
extraFeatures = np.load(open('train_feats_taskA_og.npy','rb'), allow_pickle=True)

print(f"Handcrafted OG features shape: {extraFeatures.shape}")

# Combine OG handcrafted features with w2v+emoji features
X = np.hstack([w2v_features, extraFeatures])

print(f"Final feature shape (all features): {X.shape}")

class_counts = np.asarray(np.unique(y, return_counts=True)).T.tolist()
print (f"Original class counts: {class_counts}")

Loading generic features for training...
Generic features shape: (2566, 58)
Final feature shape (all features): (2566, 58)
Original class counts: [[0, 1659], [1, 907]]


In [4]:
# IMPROVEMENT #1: Use class weights instead of undersampling
# Class weights automatically penalize mistakes on minority class without losing majority class data
y = np.array(y)  # Convert list to numpy array
print(f"Using full training data with class_weight='balanced' in SVM")
print(f"Class distribution: {class_counts}")

# Keep original data proportions
X_balanced = X  
y_balanced = y
print(f"Training samples: {len(y_balanced)} (full dataset)")

Using full training data with class_weight='balanced' in SVM
Class distribution: [[0, 1659], [1, 907]]
Training samples: 2566 (full dataset)


In [5]:
# 10-Fold Cross-Validation on balanced training data
predicted = cross_val_predict(CLF, X_balanced, y_balanced, cv=K_FOLDS)

# Modify F1-score calculation depending on the task
if TASK.lower() == 'a':
    score = metrics.f1_score(y_balanced, predicted, pos_label=1)
    p = metrics.precision_score(y_balanced, predicted, pos_label=1)
    r = metrics.recall_score(y_balanced, predicted, pos_label=1)
    acc = metrics.accuracy_score(y_balanced, predicted)
elif TASK.lower() == 'b':
    # if you set average to None, it will return results for each class separately 
    score = metrics.f1_score(y_balanced, predicted, average=None)
    score_ = metrics.f1_score(y_balanced, predicted, average='macro')
    p = metrics.precision_score(y_balanced, predicted, average="macro")
    r = metrics.recall_score(y_balanced, predicted, average="macro")
    acc = metrics.accuracy_score(y_balanced, predicted)

print("\n=== 10-Fold CV Results (BALANCED TRAINING DATA) ===")
print (f"F1-score Task {TASK}: {score}")
print (f"Precision Task {TASK}: {p}")
print (f"Recall Task {TASK}: {r}")
print (f"Accuracy Task {TASK}: {acc}")

for pred_val in predicted:
    PREDICTIONSFILE.write("{}\n".format(pred_val))
PREDICTIONSFILE.close()


=== 10-Fold CV Results (BALANCED TRAINING DATA) ===
F1-score Task A: 0.1001926782273603
Precision Task A: 0.3969465648854962
Recall Task A: 0.05733186328555678
Accuracy Task A: 0.6360093530787218


In [6]:
print("Fit on the whole balanced Training data ...")
CLF.fit(X_balanced, y_balanced)
print("Training complete.")

Fit on the whole balanced Training data ...
Training complete.


In [7]:
# save the model 
import pickle
filename = 'finalized_model_og.sav'
pickle.dump(CLF, open(filename, 'wb'))
print(f"Model saved to {filename}")

## if later you want to load the model, execute the following
# loaded_model = pickle.load(open(filename, 'rb'))

Model saved to finalized_model_og.sav


In [ ]:
print("Ready to TEST")

test_corpus, y_test = parse_dataset('../../datasets/slovene/goldtest_TaskA/SemEval2018-T3_gold_test_taskA_emoji.txt')

print("Loading test features...")
# Load handcrafted test features generated by feature_generator_TaskA_og
test_features = np.load(open('./test_feats_taskA_og.npy', 'rb'), allow_pickle=True)

print(f"Test corpus size: {len(test_corpus)}")
print(f"Test handcrafted features shape: {test_features.shape}")

# Handle mismatch in sample counts
min_test_len = min(len(test_corpus), len(test_features))
print(f"Using minimum length: {min_test_len}")

# Trim to matching length
y_test = y_test[:min_test_len]
test_features = test_features[:min_test_len]
test_corpus = test_corpus[:min_test_len]

print("Building w2v+emoji features for testing...")
w2v_features_test = build_w2v_mean(test_corpus, w2v_model, emoji_model)
print(f"Test W2V features shape: {w2v_features_test.shape}")

# Combine handcrafted OG features with w2v+emoji features
X_test = np.hstack([w2v_features_test, test_features])

print(f"Test features shape (all features): {X_test.shape}")

Ready to TEST
Loading test features...
Test corpus size: 640
Test features shape: (641, 58)
Using minimum length: 640
Test features shape (all features): (640, 58)


In [9]:
# Test evaluation with optimal threshold tuning
y_test_proba = CLF.predict_proba(X_test)

# Try multiple thresholds to find the best one
thresholds_to_test = [0.3, 0.4, 0.5, 0.6, 0.7]
print("\n=== Testing Multiple Thresholds ===")
best_f1 = 0
best_threshold = 0.5

for threshold in thresholds_to_test:
    y_pred = (y_test_proba[:, 1] >= threshold).astype(int)
    f1 = metrics.f1_score(y_test, y_pred, pos_label=1)
    prec = metrics.precision_score(y_test, y_pred, pos_label=1, zero_division=0)
    rec = metrics.recall_score(y_test, y_pred, pos_label=1, zero_division=0)
    pred_count = np.sum(y_pred)
    print(f"Threshold {threshold}: F1={f1:.4f}, Precision={prec:.4f}, Recall={rec:.4f}, Predicted Ironic={pred_count}/640")
    
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = threshold

# Use the best threshold for final predictions
THRESHOLD = best_threshold
y_test_predicted = (y_test_proba[:, 1] >= THRESHOLD).astype(int)

print(f"\nBest threshold selected: {THRESHOLD}")
print(f"Predictions: {np.unique(y_test_predicted, return_counts=True)}")

with open('predictions-taskA_og.txt', 'w') as f:
    for yp in y_test_predicted:
        f.write(str(yp)+"\n")

score = metrics.f1_score(y_test, y_test_predicted, pos_label=1)
p = metrics.precision_score(y_test, y_test_predicted, pos_label=1, zero_division=0)
r = metrics.recall_score(y_test, y_test_predicted, pos_label=1, zero_division=0)
acc = metrics.accuracy_score(y_test, y_test_predicted)

print (f"\n=== Test Set Results (threshold={THRESHOLD}) ===")
print (f"F1-score Task {TASK}: {score}")
print (f"Precision Task {TASK}: {p}")
print (f"Recall Task {TASK}: {r}")
print (f"Accuracy Task {TASK}: {acc}")


=== Testing Multiple Thresholds ===
Threshold 0.3: F1=0.5007, Precision=0.3573, Recall=0.8363, Predicted Ironic=529/640
Threshold 0.4: F1=0.4145, Precision=0.5000, Recall=0.3540, Predicted Ironic=160/640
Threshold 0.5: F1=0.0938, Precision=0.4000, Recall=0.0531, Predicted Ironic=30/640
Threshold 0.6: F1=0.0087, Precision=0.2000, Recall=0.0044, Predicted Ironic=5/640
Threshold 0.7: F1=0.0000, Precision=0.0000, Recall=0.0000, Predicted Ironic=0/640

Best threshold selected: 0.3
Predictions: (array([0, 1]), array([111, 529], dtype=int64))

=== Test Set Results (threshold=0.3) ===
F1-score Task A: 0.5006622516556293
Precision Task A: 0.3572778827977316
Recall Task A: 0.8362831858407079
Accuracy Task A: 0.4109375


In [ ]:
import pickle
import re
from collections import Counter

# Load the trained model
model = pickle.load(open('finalized_model_og.sav', 'rb'))


def extract_enhanced_features(tweet_text):
    from ekphrasis.utils.nlp import polarity
    import re

    words = tweet_text.split()
    words_lower = [w.lower() for w in words]

    # SLOVENE SENTIMENT WORDS
    positive_words = ['rad', 'obožujem', 'lepo', 'super', 'odličko', 'hvala', 'rabi', 'rabu',
                      'super', 'great', 'love', 'perfect', 'amazing', 'wonderful', 'fantastic']
    negative_words = ['zaprta', 'zaprto', 'zaprte', 'slabo', 'čudno', 'groza', 'strašno',
                      'problematično', 'dead', 'horrible', 'ugly', 'terrible', 'bad']

    # 1) Positive-negative contradiction
    positive_count = sum(1 for w in words_lower if any(w.startswith(p) or p in w for p in positive_words))
    negative_count = sum(1 for w in words_lower if any(w.startswith(n) or n in w for n in negative_words))
    sarcasm_contrast = 1 if (positive_count > 0 and negative_count > 0) else 0

    # 2) Left-right intensity/contrast
    left_half = words[:len(words)//2]
    right_half = words[len(words)//2:]
    left_word_lens = [len(w) for w in left_half]
    right_word_lens = [len(w) for w in right_half]

    left_intensity = 1 if (sum(left_word_lens) / max(len(left_half), 1)) < 4 else 0
    right_intensity = 1 if (sum(right_word_lens) / max(len(right_half), 1)) < 4 else 0
    polarity_diff = 1 if len(left_half) > 0 and len(right_half) > 0 else 0

    contrast = 0
    try:
        if len(left_half) > 0 and len(right_half) > 0:
            left_text = ' '.join(left_half)
            right_text = ' '.join(right_half)
            left_pol = polarity(left_text)
            right_pol = polarity(right_text)
            if (left_pol and right_pol and len(left_pol) > 1 and len(right_pol) > 1):
                contrast = 1 if abs(left_pol[0] - right_pol[0]) > 0.5 else 0
    except:
        pass

    # 3) Punctuation/linguistic markers
    exclamation_count = tweet_text.count('!')
    question_count = tweet_text.count('?')
    ellipsis_count = tweet_text.count('...')
    ellipsis_signal = 1 if ellipsis_count > 0 else 0
    excessive_punct = 1 if (exclamation_count > 2 or question_count > 2) else 0

    elongated_words = len([w for w in words if re.search(r'(.)\1{2,}', w)])
    elongation_score = min(elongated_words / max(len(words), 1), 1.0)

    slovene_negations = ['ne', 'nema', 'nimam', 'nimajo', 'nič', 'nikoli', 'nobeden', 'brez']
    negation_count = sum(1 for word in words_lower if any(word.startswith(neg) or neg in word for neg in slovene_negations))
    negation_score = min(negation_count / max(len(words), 1), 1.0)

    # EXACT ORIGINAL 58-DIMENSION BUILDER (NO OVERWRITING, NO APPENDING)
    core = [left_intensity, right_intensity, polarity_diff, contrast]

    aux_54 = np.zeros(54, dtype=float)
    aux_54[0] = exclamation_count / 5
    aux_54[1] = question_count / 5
    aux_54[2] = ellipsis_count / 3
    aux_54[3] = excessive_punct
    aux_54[4] = elongation_score
    aux_54[5] = negation_score
    aux_54[6] = sarcasm_contrast
    aux_54[7] = ellipsis_signal
    aux_54[8] = positive_count / max(len(words), 1)
    aux_54[9] = negative_count / max(len(words), 1)

    # Note: In OG base features, we strictly leave slots 52 & 53 UNMODIFIED
    # No emoji sentiment prior
    # No hashtag sentiment prior
    # No pragmatic heuristics (no +2 dimensions at the end)

    all_feats = core + aux_54.tolist() # Length: 4 + 54 = 58
    return all_feats


def build_combined_features(tweet_text):
    handcrafted = np.array(extract_enhanced_features(tweet_text), dtype=float)
    w2v_feat = build_w2v_mean([tweet_text], w2v_model, emoji_model)[0]
    return np.hstack([w2v_feat, handcrafted])


def predict_irony(sentence):
    feats = build_combined_features(sentence).reshape(1, -1)
    proba = model.predict_proba(feats)[0]
    return {
        "is_ironic": bool(proba[1] >= 0.5),
        "confidence_ironic": proba[1],
        "confidence_non_ironic": proba[0]
    }

# Debug: Check feature extraction for manual test
my_tweet = "tocno to sm rabu, da je deponija zaprta ko pridem tja... "

# Extract and display features
feats = extract_enhanced_features(my_tweet)
combined_feats = build_combined_features(my_tweet)
print("\n" + "="*60)
print("DEBUG: Feature Extraction Analysis")
print("="*60)
print(f"Tweet: {my_tweet}")
print(f"\nHandcrafted features extracted: {len(feats)}")
print(f"Combined features extracted: {len(combined_feats)}")
print("Key sarcasm indicators:")
print(f"  - Ellipsis signal (feat 11): {feats[11]}")
print(f"  - Positive-negative contrast (feat 10): {feats[10]}")
print(f"  - Positive word density (feat 12): {feats[12]}")
print(f"  - Negative word density (feat 13): {feats[13]}")

result = predict_irony(my_tweet)
print("\n" + "="*60)
print("PREDICTION")
print("="*60)
print(f"Result: {'IRONIČEN' if result['is_ironic'] else '✓ NI IRONIČEN'}")
print(f"Confidence: {max(result['confidence_ironic'], result['confidence_non_ironic'])*100:.2f}%")
print(f"  - Irony: {result['confidence_ironic']*100:.2f}%")
print(f"  - Not: {result['confidence_non_ironic']*100:.2f}%")
print("="*60)

In [14]:
def predict_irony(sentence):
    feats = np.array(extract_enhanced_features(sentence)).reshape(1, -1)
    proba = model.predict_proba(feats)[0]
    return {
        "is_ironic": bool(proba[1] >= 0.5),
        "confidence_ironic": proba[1],
        "confidence_non_ironic": proba[0]
    }

# Debug: Check feature extraction for manual test
my_tweet = "tocno to sm rabu, da je deponija zaprta ko pridem tja... "

# Extract and display features
feats = extract_enhanced_features(my_tweet)
print("\n" + "="*60)
print("DEBUG: Feature Extraction Analysis")
print("="*60)
print(f"Tweet: {my_tweet}")
print(f"\nTotal features extracted: {len(feats)}")
print("Key sarcasm indicators:")
print(f"  - Ellipsis signal (feat 11): {feats[11]}")
print(f"  - Positive-negative contrast (feat 10): {feats[10]}")
print(f"  - Positive word density (feat 12): {feats[12]}")
print(f"  - Negative word density (feat 13): {feats[13]}")

result = predict_irony(my_tweet)
print("\n" + "="*60)
print("PREDICTION")
print("="*60)
print(f"Result: {'IRONIČEN' if result['is_ironic'] else '✓ NI IRONIČEN'}")
print(f"Confidence: {max(result['confidence_ironic'], result['confidence_non_ironic'])*100:.2f}%")
print(f"  - Irony: {result['confidence_ironic']*100:.2f}%")
print(f"  - Not: {result['confidence_non_ironic']*100:.2f}%")
print("="*60)


DEBUG: Feature Extraction Analysis
Tweet: tocno to sm rabu, da je deponija zaprta ko pridem tja... 

Total features extracted: 58
Key sarcasm indicators:
  - Ellipsis signal (feat 11): 1.0
  - Positive-negative contrast (feat 10): 1.0
  - Positive word density (feat 12): 0.09090909090909091
  - Negative word density (feat 13): 0.09090909090909091

PREDICTION
Result: IRONIČEN
Confidence: 57.20%
  - Irony: 57.20%
  - Not: 42.80%


In [ ]:
# Compare near-identical tweets to inspect baseline behavior
compare_tweets = [
    "še dobro da Dve uri čakanja v čakalnici, da mi rečejo, da sem naročen naslednji teden. Moj čas očitno nima nobene vrednosti, hvala za to izkušnjo! 🙃",
    "Vrhunsko. Dve uri čakanja v čakalnici, da mi rečejo, da sem naročen naslednji teden. Moj čas očitno nima nobene vrednosti, hvala za to izkušnjo! 🙃"
]

print("\n" + "="*70)
print("PAIRWISE CHECK: BASELINE MODEL PREDICTIONS (58-DIM + W2V + EMOJI)")
print("="*70)
for tw in compare_tweets:
    handcrafted = extract_enhanced_features(tw)
    combined = build_combined_features(tw)
    res = predict_irony(tw)
    print(f"\nTweet: {tw}")
    print(f"Handcrafted dims: {len(handcrafted)}")
    print(f"Combined dims: {len(combined)}")
    print(f"Prediction: {'IRONIC' if res['is_ironic'] else 'NOT IRONIC'}")
    print(f"Confidence: {max(res['confidence_ironic'], res['confidence_non_ironic']):.4f}")
    print(f"  Irony prob: {res['confidence_ironic']:.4f}")
    print(f"  Non-irony prob: {res['confidence_non_ironic']:.4f}")
    print("-"*70)


PAIRWISE CHECK: BASELINE MODEL PREDICTIONS (58-DIM)

Tweet: še dobro da Dve uri čakanja v čakalnici, da mi rečejo, da sem naročen naslednji teden. Moj čas očitno nima nobene vrednosti, hvala za to izkušnjo! 🙃
Prediction: IRONIC
Confidence: 0.5444
  Irony prob: 0.5444
  Non-irony prob: 0.4556
----------------------------------------------------------------------

Tweet: Vrhunsko. Dve uri čakanja v čakalnici, da mi rečejo, da sem naročen naslednji teden. Moj čas očitno nima nobene vrednosti, hvala za to izkušnjo! 🙃
Prediction: NOT IRONIC
Confidence: 0.5791
  Irony prob: 0.4209
  Non-irony prob: 0.5791
----------------------------------------------------------------------


In [ ]:
# Full feature dump + linear contribution proxy for one tweet
analysis_tweet = "Vrhunsko. Dve uri čakanja v čakalnici, da mi rečejo, da sem naročen naslednji teden. Moj čas očitno nima nobene vrednosti, hvala za to izkušnjo! 🙃"

handcrafted_vals = np.array(extract_enhanced_features(analysis_tweet), dtype=float)
combined_vals = build_combined_features(analysis_tweet)
proba = model.predict_proba(combined_vals.reshape(1, -1))[0]

# Build readable names for all combined features
w2v_dim = combined_vals.shape[0] - handcrafted_vals.shape[0]
feature_names = [f"w2v_{i}" for i in range(w2v_dim)]
feature_names += [
    "left_intensity", "right_intensity", "polarity_diff", "contrast"
]
feature_names += [f"aux_{i}" for i in range(54)]

# Alias important handcrafted slots
aliases = {
    w2v_dim + 4: "exclamation_norm", w2v_dim + 5: "question_norm", w2v_dim + 6: "ellipsis_norm", w2v_dim + 7: "excessive_punct",
    w2v_dim + 8: "elongation_score", w2v_dim + 9: "negation_score", w2v_dim + 10: "sarcasm_contrast", w2v_dim + 11: "ellipsis_signal",
    w2v_dim + 12: "positive_density", w2v_dim + 13: "negative_density", w2v_dim + 56: "emoji_sentiment_prior", w2v_dim + 57: "hashtag_sentiment_prior"
}

print("=" * 90)
print("FULL FEATURE DUMP")
print("=" * 90)
print("Tweet:")
print(analysis_tweet)
print("-" * 90)
print(f"Model probabilities -> non_ironic={proba[0]:.4f}, ironic={proba[1]:.4f}")
print(f"Combined feature length: {combined_vals.shape[0]} (w2v={w2v_dim} + handcrafted={handcrafted_vals.shape[0]})")
print("=" * 90)

for i, val in enumerate(combined_vals):
    name = aliases.get(i, feature_names[i])
    print(f"{i:02d} {name:28s} = {val: .6f}")

# Contribution proxy using the LR member inside VotingClassifier
print("\n" + "=" * 90)
print("TOP LR CONTRIBUTION PROXY (coef * value)")
print("=" * 90)
try:
    lr_member = CLF.estimators_[1]
    lr_coef = lr_member.coef_[0]
    contrib = lr_coef * combined_vals
    top_idx = np.argsort(np.abs(contrib))[::-1][:15]

    for idx in top_idx:
        name = aliases.get(idx, feature_names[idx])
        direction = "toward ironic" if contrib[idx] > 0 else "toward non-ironic"
        print(
            f"{idx:02d} {name:28s} "
            f"value={combined_vals[idx]: .6f} coef={lr_coef[idx]: .6f} "
            f"contrib={contrib[idx]: .6f} ({direction})"
        )
except Exception as e:
    print(f"Could not compute LR contribution proxy: {e}")

FULL FEATURE DUMP
Tweet:
Vrhunsko. Dve uri čakanja v čakalnici, da mi rečejo, da sem naročen naslednji teden. Moj čas očitno nima nobene vrednosti, hvala za to izkušnjo! 🙃
------------------------------------------------------------------------------------------
Model probabilities -> non_ironic=0.5791, ironic=0.4209
00 left_intensity               =  0.000000
01 right_intensity              =  0.000000
02 polarity_diff                =  1.000000
03 contrast                     =  0.000000
04 exclamation_norm             =  0.200000
05 question_norm                =  0.000000
06 ellipsis_norm                =  0.000000
07 excessive_punct              =  0.000000
08 elongation_score             =  0.000000
09 negation_score               =  0.040000
10 sarcasm_contrast             =  0.000000
11 ellipsis_signal              =  0.000000
12 positive_density             =  0.040000
13 negative_density             =  0.000000
14 aux_10                       =  0.000000
15 aux_11            

In [ ]:
# Compact diagnostic: only non-zero features + strongest LR contributions
analysis_tweet_small = "Vrhunsko. Dve uri čakanja v čakalnici, da mi rečejo, da sem naročen naslednji teden. Moj čas očitno nima nobene vrednosti, hvala za to izkušnjo! 🙃"
handcrafted_small = np.array(extract_enhanced_features(analysis_tweet_small), dtype=float)
combined_small = build_combined_features(analysis_tweet_small)
proba_small = model.predict_proba(combined_small.reshape(1, -1))[0]

w2v_dim = combined_small.shape[0] - handcrafted_small.shape[0]
alias = {
    0: "w2v_0", 1: "w2v_1", 2: "w2v_2", 3: "w2v_3",
    w2v_dim + 0: "left_intensity", w2v_dim + 1: "right_intensity", w2v_dim + 2: "polarity_diff", w2v_dim + 3: "contrast",
    w2v_dim + 4: "exclamation_norm", w2v_dim + 5: "question_norm", w2v_dim + 6: "ellipsis_norm", w2v_dim + 7: "excessive_punct",
    w2v_dim + 8: "elongation_score", w2v_dim + 9: "negation_score", w2v_dim + 10: "sarcasm_contrast", w2v_dim + 11: "ellipsis_signal",
    w2v_dim + 12: "positive_density", w2v_dim + 13: "negative_density", w2v_dim + 52: "emoji_sentiment_prior",
    w2v_dim + 53: "hashtag_sentiment_prior", w2v_dim + 54: "emoji_irony_soft_prior", w2v_dim + 55: "inconvenience_soft_prior"
}

print("\n" + "="*70)
print("COMPACT FEATURE REASONING")
print("="*70)
print(f"non_ironic={proba_small[0]:.4f}, ironic={proba_small[1]:.4f}")
print(f"Combined feature length: {combined_small.shape[0]} (w2v={w2v_dim} + handcrafted={handcrafted_small.shape[0]})")
print("\nNon-zero features:")
for i, v in enumerate(combined_small):
    if abs(v) > 1e-9:
        print(f"{i:02d} {alias.get(i, f'feat_{i}')} = {v:.6f}")

print("\nTop LR contributions (coef*value):")
lr_member = CLF.estimators_[1]
coef = lr_member.coef_[0]
contrib = coef * combined_small
top = np.argsort(np.abs(contrib))[::-1][:12]
for idx in top:
    if abs(contrib[idx]) < 1e-9:
        continue
    direction = "ironic" if contrib[idx] > 0 else "non-ironic"
    print(f"{idx:02d} {alias.get(idx, f'feat_{idx}')}: contrib={contrib[idx]:.6f} -> {direction}")


COMPACT FEATURE REASONING
non_ironic=0.5791, ironic=0.4209

Non-zero features:
02 polarity_diff = 1.000000
04 exclamation_norm = 0.200000
09 negation_score = 0.040000
12 positive_density = 0.040000

Top LR contributions (coef*value):
02 polarity_diff: contrib=0.782061 -> ironic
09 negation_score: contrib=-0.010380 -> non-ironic
04 exclamation_norm: contrib=-0.006768 -> non-ironic
12 positive_density: contrib=-0.006565 -> non-ironic
